In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Week3_Order_Processing").getOrCreate()
print("Spark Session created")

Spark Session created


In [10]:
df = spark.read.option("header", True).option("inferSchema", True).csv("sample_supply_chain_data.csv")

In [11]:
df.show(truncate=False)

+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+---------------------+
|order_id|supplier_id|product_name|order_date|delivery_date|ordered_qty|stock_on_hand|unit_price|status   |priority|notes                |
+--------+-----------+------------+----------+-------------+-----------+-------------+----------+---------+--------+---------------------+
|101     |1          |Rice        |2026-06-01|2026-06-05   |50         |40           |32.5      |Delivered|High    |On time              |
|102     |2          |Wheat       |2026-06-02|2026-06-08   |30         |18           |28.0      |Delivered|Medium  |Late due to transport|
|103     |3          |Sugar       |2026-06-03|NULL         |20         |25           |41.0      |Pending  |Low     |NULL                 |
|104     |1          |Salt        |2026-06-04|2026-06-06   |10         |12           |15.0      |Delivered|High    |Fast delivery        |
|105     |4          |Oil  

In [12]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- ordered_qty: integer (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- notes: string (nullable = true)



In [13]:
df = df.na.fill(
    {
        "notes": "No notes",
        "status": "Pending",
        "priority": "Medium"
    })

In [14]:
df = df.withColumn("order_date", to_timestamp(col("order_date"), "yyyy-MM-dd")) \
       .withColumn("delivery_date", to_timestamp(col("delivery_date"), "yyyy-MM-dd"))

In [15]:
df = df.withColumn(
    "delivery_date",
    when(col("delivery_date").isNull(), current_timestamp()).otherwise(col("delivery_date"))
)

df.show(truncate=False)

+--------+-----------+------------+-------------------+--------------------------+-----------+-------------+----------+---------+--------+---------------------+
|order_id|supplier_id|product_name|order_date         |delivery_date             |ordered_qty|stock_on_hand|unit_price|status   |priority|notes                |
+--------+-----------+------------+-------------------+--------------------------+-----------+-------------+----------+---------+--------+---------------------+
|101     |1          |Rice        |2026-06-01 00:00:00|2026-06-05 00:00:00       |50         |40           |32.5      |Delivered|High    |On time              |
|102     |2          |Wheat       |2026-06-02 00:00:00|2026-06-08 00:00:00       |30         |18           |28.0      |Delivered|Medium  |Late due to transport|
|103     |3          |Sugar       |2026-06-03 00:00:00|2026-06-27 04:54:19.724796|20         |25           |41.0      |Pending  |Low     |No notes             |
|104     |1          |Salt        

In [29]:
df.filter((col("priority") == "High") & (col("is_delayed") == 1)).show()

+--------+-----------+------------+-------------------+--------------------+-----------+-------------+----------+---------+--------+--------------------+----------+----------+---------+
|order_id|supplier_id|product_name|         order_date|       delivery_date|ordered_qty|stock_on_hand|unit_price|   status|priority|               notes|delay_days|is_delayed|stock_gap|
+--------+-----------+------------+-------------------+--------------------+-----------+-------------+----------+---------+--------+--------------------+----------+----------+---------+
|     101|          1|        Rice|2026-06-01 00:00:00| 2026-06-05 00:00:00|         50|           40|      32.5|Delivered|    High|             On time|         4|         1|       10|
|     107|          2|      Coffee|2026-06-07 00:00:00| 2026-06-12 00:00:00|         25|           20|     310.0|Delivered|    High|               Delay|         5|         1|        5|
|     110|          5|        Soap|2026-06-10 00:00:00|2026-06-27 04:5

In [16]:
df = df.withColumn("delay_days", datediff(col("delivery_date"), col("order_date"))) \
       .withColumn("is_delayed", when(col("delay_days") > 3, 1).otherwise(0))

df.show()

+--------+-----------+------------+-------------------+--------------------+-----------+-------------+----------+---------+--------+--------------------+----------+----------+
|order_id|supplier_id|product_name|         order_date|       delivery_date|ordered_qty|stock_on_hand|unit_price|   status|priority|               notes|delay_days|is_delayed|
+--------+-----------+------------+-------------------+--------------------+-----------+-------------+----------+---------+--------+--------------------+----------+----------+
|     101|          1|        Rice|2026-06-01 00:00:00| 2026-06-05 00:00:00|         50|           40|      32.5|Delivered|    High|             On time|         4|         1|
|     102|          2|       Wheat|2026-06-02 00:00:00| 2026-06-08 00:00:00|         30|           18|      28.0|Delivered|  Medium|Late due to trans...|         6|         1|
|     103|          3|       Sugar|2026-06-03 00:00:00|2026-06-27 04:54:...|         20|           25|      41.0|  Pendi

In [17]:
delayed_df = df.filter(col("is_delayed") == 1)
delayed_df.show(truncate=False)

+--------+-----------+------------+-------------------+--------------------------+-----------+-------------+----------+---------+--------+---------------------+----------+----------+
|order_id|supplier_id|product_name|order_date         |delivery_date             |ordered_qty|stock_on_hand|unit_price|status   |priority|notes                |delay_days|is_delayed|
+--------+-----------+------------+-------------------+--------------------------+-----------+-------------+----------+---------+--------+---------------------+----------+----------+
|101     |1          |Rice        |2026-06-01 00:00:00|2026-06-05 00:00:00       |50         |40           |32.5      |Delivered|High    |On time              |4         |1         |
|102     |2          |Wheat       |2026-06-02 00:00:00|2026-06-08 00:00:00       |30         |18           |28.0      |Delivered|Medium  |Late due to transport|6         |1         |
|103     |3          |Sugar       |2026-06-03 00:00:00|2026-06-27 04:54:21.822657|20 

In [18]:
grouped_df = delayed_df.groupBy("supplier_id").agg(
    count("order_id").alias("delayed_order_count")
).orderBy(desc("delayed_order_count"))

grouped_df.show(truncate=False)

+-----------+-------------------+
|supplier_id|delayed_order_count|
+-----------+-------------------+
|3          |2                  |
|2          |2                  |
|1          |1                  |
|5          |1                  |
|4          |1                  |
+-----------+-------------------+



In [19]:
df.agg(sum("ordered_qty").alias("total_ordered_qty")).show()

+-----------------+
|total_ordered_qty|
+-----------------+
|              303|
+-----------------+



In [20]:
df.agg( avg("delay_days").alias("avg_delay_days")).show()

+--------------+
|avg_delay_days|
+--------------+
|           7.3|
+--------------+



In [21]:
 df.agg(count("order_id").alias("total_orders")).show()

+------------+
|total_orders|
+------------+
|          10|
+------------+



In [22]:
df.orderBy(col("delay_days").desc()).show()

+--------+-----------+------------+-------------------+--------------------+-----------+-------------+----------+---------+--------+--------------------+----------+----------+
|order_id|supplier_id|product_name|         order_date|       delivery_date|ordered_qty|stock_on_hand|unit_price|   status|priority|               notes|delay_days|is_delayed|
+--------+-----------+------------+-------------------+--------------------+-----------+-------------+----------+---------+--------+--------------------+----------+----------+
|     103|          3|       Sugar|2026-06-03 00:00:00|2026-06-27 04:54:...|         20|           25|      41.0|  Pending|     Low|            No notes|        24|         1|
|     110|          5|        Soap|2026-06-10 00:00:00|2026-06-27 04:54:...|         35|           30|      25.0|  Pending|    High|Waiting for shipment|        17|         1|
|     102|          2|       Wheat|2026-06-02 00:00:00| 2026-06-08 00:00:00|         30|           18|      28.0|Deliver

In [23]:
df.select("supplier_id").distinct().show()

+-----------+
|supplier_id|
+-----------+
|          1|
|          3|
|          5|
|          4|
|          2|
+-----------+



In [24]:
df.select("order_id", "supplier_id", "product_name", "delay_days", "is_delayed").show()

+--------+-----------+------------+----------+----------+
|order_id|supplier_id|product_name|delay_days|is_delayed|
+--------+-----------+------------+----------+----------+
|     101|          1|        Rice|         4|         1|
|     102|          2|       Wheat|         6|         1|
|     103|          3|       Sugar|        24|         1|
|     104|          1|        Salt|         2|         0|
|     105|          4|         Oil|         5|         1|
|     106|          5|         Tea|         3|         0|
|     107|          2|      Coffee|         5|         1|
|     108|          3| Milk Powder|         5|         1|
|     109|          4|     Biscuit|         2|         0|
|     110|          5|        Soap|        17|         1|
+--------+-----------+------------+----------+----------+



In [25]:
df_renamed = df.withColumnRenamed("ordered_qty", "quantity_ordered")
df_renamed.show()

+--------+-----------+------------+-------------------+--------------------+----------------+-------------+----------+---------+--------+--------------------+----------+----------+
|order_id|supplier_id|product_name|         order_date|       delivery_date|quantity_ordered|stock_on_hand|unit_price|   status|priority|               notes|delay_days|is_delayed|
+--------+-----------+------------+-------------------+--------------------+----------------+-------------+----------+---------+--------+--------------------+----------+----------+
|     101|          1|        Rice|2026-06-01 00:00:00| 2026-06-05 00:00:00|              50|           40|      32.5|Delivered|    High|             On time|         4|         1|
|     102|          2|       Wheat|2026-06-02 00:00:00| 2026-06-08 00:00:00|              30|           18|      28.0|Delivered|  Medium|Late due to trans...|         6|         1|
|     103|          3|       Sugar|2026-06-03 00:00:00|2026-06-27 04:54:...|              20|  

In [26]:
df = df.withColumn(
    "stock_gap",
    when(col("stock_on_hand") < col("ordered_qty"),
         col("ordered_qty") - col("stock_on_hand")).otherwise(0)
)

df.show()

+--------+-----------+------------+-------------------+--------------------+-----------+-------------+----------+---------+--------+--------------------+----------+----------+---------+
|order_id|supplier_id|product_name|         order_date|       delivery_date|ordered_qty|stock_on_hand|unit_price|   status|priority|               notes|delay_days|is_delayed|stock_gap|
+--------+-----------+------------+-------------------+--------------------+-----------+-------------+----------+---------+--------+--------------------+----------+----------+---------+
|     101|          1|        Rice|2026-06-01 00:00:00| 2026-06-05 00:00:00|         50|           40|      32.5|Delivered|    High|             On time|         4|         1|       10|
|     102|          2|       Wheat|2026-06-02 00:00:00| 2026-06-08 00:00:00|         30|           18|      28.0|Delivered|  Medium|Late due to trans...|         6|         1|       12|
|     103|          3|       Sugar|2026-06-03 00:00:00|2026-06-27 04:5

In [27]:
print("Total records:", df.count())

Total records: 10


In [28]:
df.write.mode("overwrite").parquet("processed_supply_chain_parquet")